In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
import statsmodels.graphics.tsaplots
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import statistics

In [5]:
data = pd.read_csv('Data/preprocessed/NP15_rt_series.csv')
data['begin_time'] = pd.to_datetime(data['begin_time'])
data


,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price
0,2021-01-01 00:00:00,30.76070,NaN,NaN
1,2021-01-01 01:00:00,29.89887,NaN,NaN
2,2021-01-01 02:00:00,28.47212,NaN,NaN
3,2021-01-01 03:00:00,29.79934,NaN,NaN
4,2021-01-01 04:00:00,29.26075,NaN,NaN
...,...,...,...,...
35059,2024-12-31 19:00:00,47.43169,9.47854,37.95315
35060,2024-12-31 20:00:00,46.46807,7.75172,38.71635
35061,2024-12-31 21:00:00,44.37653,3.38276,40.99377
35062,2024-12-31 22:00:00,46.56593,2.44625,44.11968


In [4]:
exog = pd.read_csv('Data/preprocessed/NP15_exog.csv')
exog['begin_time'] = pd.to_datetime(exog['begin_time'])
exog

,begin_time,load,Pacific Gas and Electric Forecast Load (MW),solar_forecast,wind_forecast,NP15 Solar Generation (MW),NP15 Wind Generation (MW),Natural_gas_price
0,2021-03-12 00:00:00,9914.0,10102.72,0.0,327.97,-2.69832,526.78168,2.65
1,2021-03-12 01:00:00,9827.0,9868.57,0.0,259.94,-2.66009,405.97186,2.65
2,2021-03-12 02:00:00,9849.0,9712.50,0.0,191.44,-2.68074,288.43194,2.65
3,2021-03-12 03:00:00,9939.0,9715.76,0.0,148.07,-2.68692,205.85381,2.65
4,2021-03-12 04:00:00,10287.0,9969.51,0.0,134.36,-2.58009,173.78675,2.65
...,...,...,...,...,...,...,...,...
33379,2024-12-31 19:00:00,11307.0,11959.79,0.0,48.25,-3.03842,58.35641,3.40
33380,2024-12-31 20:00:00,10967.0,11723.02,0.0,55.64,-7.19792,65.56839,3.40
33381,2024-12-31 21:00:00,10614.0,11408.80,0.0,65.90,-6.79170,62.77812,3.40
33382,2024-12-31 22:00:00,10226.0,10831.46,0.0,77.46,-6.26466,54.79276,3.40


In [6]:
data_comb = pd.merge(data, exog, how='inner', on='begin_time')
data_comb

,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price,load,Pacific Gas and Electric Forecast Load (MW),solar_forecast,wind_forecast,NP15 Solar Generation (MW),NP15 Wind Generation (MW),Natural_gas_price
0,2021-03-12 00:00:00,34.56084,-21.12893,55.68977,9914.0,10102.72,0.0,327.97,-2.69832,526.78168,2.65
1,2021-03-12 01:00:00,33.10504,-18.19313,51.29817,9827.0,9868.57,0.0,259.94,-2.66009,405.97186,2.65
2,2021-03-12 02:00:00,34.53900,-9.97579,44.51479,9849.0,9712.50,0.0,191.44,-2.68074,288.43194,2.65
3,2021-03-12 03:00:00,33.62841,-11.80623,45.43464,9939.0,9715.76,0.0,148.07,-2.68692,205.85381,2.65
4,2021-03-12 04:00:00,34.56917,-10.34336,44.91253,10287.0,9969.51,0.0,134.36,-2.58009,173.78675,2.65
...,...,...,...,...,...,...,...,...,...,...,...
33379,2024-12-31 19:00:00,47.43169,9.47854,37.95315,11307.0,11959.79,0.0,48.25,-3.03842,58.35641,3.40
33380,2024-12-31 20:00:00,46.46807,7.75172,38.71635,10967.0,11723.02,0.0,55.64,-7.19792,65.56839,3.40
33381,2024-12-31 21:00:00,44.37653,3.38276,40.99377,10614.0,11408.80,0.0,65.90,-6.79170,62.77812,3.40
33382,2024-12-31 22:00:00,46.56593,2.44625,44.11968,10226.0,10831.46,0.0,77.46,-6.26466,54.79276,3.40


In [ ]:
data_train=data.iloc[:26707]
data_train